In [10]:
# Computes the ray coverage in a 3D grid from ray path data stored in .npy files.
import numpy as np
import os
import glob

def compute_coverage_from_rays(ray_folders, nx, ny, nz,
                                x_bounds=(-2.5, 3),
                                y_bounds=(-4, 4),
                                z_bounds=(0, 4.5),
                                mode='path_length'):
    coverage = np.zeros((nx, ny, nz), dtype=float)
    total_rays = 0

    x_min, x_max = x_bounds
    y_min, y_max = y_bounds
    z_min, z_max = z_bounds

    for folder in ray_folders:
        files = sorted(glob.glob(os.path.join(folder, '*.npy')))
        print(f"Found {len(files)} ray files in {folder}")

        for fpath in files:
            pts = np.load(fpath)

            gx = (pts[:, 0] - x_min) / (x_max - x_min) * nx
            gy = (pts[:, 1] - y_min) / (y_max - y_min) * ny
            gz = (pts[:, 2] - z_min) / (z_max - z_min) * nz

            grid_pts = np.stack([gx, gy, gz], axis=1)
            midpoints = 0.5 * (grid_pts[:-1] + grid_pts[1:])
            seg_lengths = np.linalg.norm(np.diff(pts, axis=0), axis=1)

            ix = np.floor(midpoints[:, 0]).astype(int)
            iy = np.floor(midpoints[:, 1]).astype(int)
            iz = np.floor(midpoints[:, 2]).astype(int)

            valid = (
                (ix >= 0) & (ix < nx) &
                (iy >= 0) & (iy < ny) &
                (iz >= 0) & (iz < nz)
            )

            flat = np.ravel_multi_index(
                (ix[valid], iy[valid], iz[valid]), (nx, ny, nz)
            )
            np.add.at(coverage.ravel(), flat, seg_lengths[valid])
            total_rays += 1

    print(f"\nDone. Processed {total_rays} rays.")
    print(f"Coverage — min: {coverage.min():.4f}, max: {coverage.max():.4f}, "
          f"non-zero cells: {np.count_nonzero(coverage)}/{nx*ny*nz}")
    return coverage

def diagnose_ray_coordinates(ray_folders):
    """
    Scan all ray files and report the global coordinate ranges.
    Compare these against your known model bounds to work out the mapping.
    """
    x_all, y_all, z_all = [], [], []

    for folder in ray_folders:
        files = sorted(glob.glob(os.path.join(folder, '*.npy')))
        for fpath in files:
            pts = np.load(fpath)
            x_all.append(pts[:, 0])
            y_all.append(pts[:, 1])
            z_all.append(pts[:, 2])

    x_all = np.concatenate(x_all)
    y_all = np.concatenate(y_all)
    z_all = np.concatenate(z_all)

    print(f"X: min={x_all.min():.4f}, max={x_all.max():.4f}")
    print(f"Y: min={y_all.min():.4f}, max={y_all.max():.4f}")
    print(f"Z: min={z_all.min():.4f}, max={z_all.max():.4f}")

# ======= EXECUTION BLOCK =======
coverage = compute_coverage_from_rays(
    ray_folders=['/home/users/exet5760/Documents/SOLA_4th_year_projects/rays_P',
                 '/home/users/exet5760/Documents/SOLA_4th_year_projects/rays_S'],
    nx=9, ny=9, nz=9,
    x_bounds=(-2.25, 2.75),
    y_bounds=(-4, 4),
    z_bounds=(0, 5),
    mode='path_length'
)

# diagnose_ray_coordinates(['/home/users/exet5760/Documents/SOLA_4th_year_projects/rays_P', '/home/users/exet5760/Documents/SOLA_4th_year_projects/rays_S'])

np.save('ray_coverage.npy', coverage)

Found 23989 ray files in /home/users/exet5760/Documents/SOLA_4th_year_projects/rays_P
Found 12696 ray files in /home/users/exet5760/Documents/SOLA_4th_year_projects/rays_S

Done. Processed 36685 rays.
Coverage — min: 0.0000, max: 14846.0000, non-zero cells: 142/729


In [ ]:
# Computes adaptive target kernels for a 3D grid based on local ray coverage, writing sparse kernel files.
import numpy as np
import os
from write_Target import compute_Tk_sphere, write_Tk_sparse

def compute_coverage_radii(coverage, radius_min, radius_max, power=0.5):
    """
    Map a 3D coverage array to per-cell radii.

    High coverage  → small radius (radius_min)
    Low coverage   → large radius (radius_max)

    Parameters
    ----------
    coverage   : np.ndarray, shape (nx, ny, nz)
    radius_min : float — smallest radius (high-coverage cells)
    radius_max : float — largest radius (low/no-coverage cells)
    power      : float — 0.5 compresses dynamic range (gentler),
                         1.0 is linear, >1 more aggressive
    """
    c = coverage.astype(float)
    c_max = c.max()
    if c_max == 0:
        return np.full(c.shape, radius_max)

    c_norm = (c / c_max) ** power       # high coverage → 1
    radii = radius_max - (radius_max - radius_min) * c_norm
    return radii


def compute_adaptive_Tks_3D(nx, ny, nz, coverage, outdir,
                             radius_min=0.5, radius_max=3.0,
                             power=0.5, tol=0.0):
    """
    Compute adaptive target kernels for every cell.

    Each cell gets a spherical kernel whose radius is determined by
    local data coverage: more coverage → smaller radius.

    Parameters
    ----------
    nx, ny, nz  : int — grid dimensions
    coverage    : np.ndarray, shape (nx, ny, nz)
    cfg         : config object with cfg.indir attribute
    radius_min  : minimum kernel radius (well-covered cells)
    radius_max  : maximum kernel radius (poorly-covered cells)
    power       : exponent controlling how aggressively radius varies
    tol         : sparsity threshold for writing
    """
    ncell = nx * ny * nz
    outdir = os.path.join(outdir, "T_adaptive")
    if not os.path.exists(outdir):
        os.mkdir(outdir)

    radii = compute_coverage_radii(coverage, radius_min, radius_max, power)

    radii_flat = radii.flatten()
    print(f"Adaptive radii — min: {radii_flat.min():.3f}, "
          f"max: {radii_flat.max():.3f}, "
          f"mean: {radii_flat.mean():.3f}")

    k = 0
    for ix in range(nx):
        for iy in range(ny):
            for iz in range(nz):
                xk = ix + 0.5
                yk = iy + 0.5
                zk = iz + 0.5

                r = radii[ix, iy, iz]

                Tk = compute_Tk_sphere(xk, yk, zk, r, nx, ny, nz)

                fname = os.path.join(outdir, f"T_{k}")
                write_Tk_sparse(Tk, ncell, fname, tol=tol)

                if k % 50 == 0:
                    print(f"Written kernel {k}/{ncell}  "
                          f"(cell [{ix},{iy},{iz}], radius={r:.3f})")
                k += 1

    # Save radius field for diagnostics
    radii_path = os.path.join(outdir, "adaptive_radii.npy")
    np.save(radii_path, radii)
    print(f"\nDone. Radius field saved to {radii_path}")

# ===== EXECUTION BLOCK ======
coverage = np.load('ray_coverage.npy').reshape(9, 9, 9)

compute_adaptive_Tks_3D(
    nx=9, ny=9, nz=9,
    coverage=coverage,
    outdir='/home/users/exet5760/Documents/SOLA_4th_year_projects/',
    radius_min=0.5,
    radius_max=3.0,
    power=0.5,
    tol=1e-8
)


Adaptive radii — min: 0.500, max: 2.000, mean: 1.945
Written kernel 0/729  (cell [0,0,0], radius=2.000)


KeyboardInterrupt: 